In [1]:
import torch
import numpy as np

In [2]:
data = [[1, 2], [3, 4]]
torch.tensor(data)

tensor([[1, 2],
        [3, 4]])

In [3]:
x = np.array(data)
torch.from_numpy(x)

tensor([[1, 2],
        [3, 4]])

In [4]:
shape = (
    2,
    3,
)

In [5]:
torch.rand(shape)

tensor([[0.8751, 0.3507, 0.6325],
        [0.5165, 0.0702, 0.1558]])

In [12]:
torch.rand(shape).reshape((3, 2))

tensor([[0.3210, 0.3336],
        [0.4889, 0.0353],
        [0.9168, 0.9169]])

In [6]:
torch.ones(shape)

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [7]:
torch.zeros(shape)

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [8]:
check = torch.rand((2, 3))
print(f"Data type: {check.dtype}, Device {check.device}")

Data type: torch.float32, Device cpu


In [10]:
tensor = check
tensor.shape

torch.Size([2, 3])

In [13]:
torch.mul(check, check)

tensor([[0.9550, 0.4063, 0.6194],
        [0.7206, 0.6539, 0.1175]])

In [15]:
x = torch.matmul(tensor, tensor.T)

In [16]:
x.shape

torch.Size([2, 2])

In [25]:
x = tensor @ tensor.T

In [27]:
y = torch.matmul(tensor, tensor.T)

In [29]:
assert (y == x).all()

In [30]:
import copy

tensor = copy.deepcopy(check)

In [31]:
tensor @ tensor.T

tensor([[1.9806, 1.6147],
        [1.6147, 1.4920]])

In [18]:
torch.matmul(tensor, tensor.T)

tensor([[0.4139, 0.2585],
        [0.2585, 0.3713]])

In [34]:
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)

In [35]:
device

'mps'

In [36]:
tensor.device

device(type='cpu')

In [37]:
tensor = tensor.to(device=device)

In [38]:
tensor.device

device(type='mps', index=0)

In [39]:
%timeit tensor@tensor.T

11.4 μs ± 58.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [41]:
tensor_cpu = tensor.to(device="cpu")

In [42]:
%timeit tensor_cpu@tensor_cpu.T

716 ns ± 4.25 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [44]:
tensor_cpu @ tensor.T

RuntimeError: Tensor for argument #1 'mat1' is on CPU, but expected it to be on GPU (while checking arguments for mm)

In [45]:
torch.cat([tensor, tensor, tensor], dim=1)

tensor([[0.9772, 0.6374, 0.7870, 0.9772, 0.6374, 0.7870, 0.9772, 0.6374, 0.7870],
        [0.8489, 0.8087, 0.3428, 0.8489, 0.8087, 0.3428, 0.8489, 0.8087, 0.3428]],
       device='mps:0')

In [46]:
class DenseLayer(torch.nn.Module):
    def __init__(self, n, d):
        super(DenseLayer, self).__init__()
        self.W = torch.nn.Parameter(torch.randn(n, d, requires_grad=True))
        self.b = torch.nn.Parameter(torch.randn(1, d, requires_grad=True))

    def forward(self, inputs):
        z = inputs @ self.W.T + self.b
        # output sigmoid ..
        return torch.sigmoid(z)

In [48]:
layer = DenseLayer(10, 10)

In [49]:
model = DenseLayer(10, 10)

In [50]:
inputs = torch.tensor(2)
model = DenseLayer(10, 10)

TypeError: DenseLayer.__init__() missing 1 required positional argument: 'd'

In [51]:
input = 2
output = 1


class MyNeuralNetwork(torch.nn.Module):
    def __init__(
        self,
    ):
        super().__init__()
        self.W = torch.nn.Parameter(torch.rand(input, output))
        self.b = torch.nn.Parameter(torch.zeros(output))

    def forward(self, inputs):
        z = torch.matmul(inputs, self.W) + self.b
        # output sigmoid ..
        return z

In [52]:
num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(
    mean=[0, 3], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)
positive_samples = np.random.multivariate_normal(
    mean=[3, 0], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)

In [53]:
negative_samples

array([[ 1.54729021,  5.6175753 ],
       [-1.7232977 ,  1.41837185],
       [-0.51889409,  3.61070341],
       ...,
       [ 1.48738426,  3.73692923],
       [-0.35258832,  2.43324741],
       [-0.27531185,  2.57660682]], shape=(1000, 2))

In [54]:
positive_samples

array([[ 3.37675661, -1.4993536 ],
       [ 2.48423348, -2.25798979],
       [ 3.41597835, -0.8911956 ],
       ...,
       [ 4.39322319, -0.10446829],
       [ 2.31017292,  0.72262933],
       [ 2.83096321, -1.06021532]], shape=(1000, 2))

In [60]:
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)

In [61]:
inputs.shape

(2000, 2)

In [62]:
model = MyNeuralNetwork()
tensor = torch.tensor(inputs)

In [63]:
output = model(tensor)

In [64]:
inputs

array([[ 1.5472902 ,  5.617575  ],
       [-1.7232977 ,  1.4183718 ],
       [-0.5188941 ,  3.6107035 ],
       ...,
       [ 4.3932233 , -0.10446829],
       [ 2.310173  ,  0.7226293 ],
       [ 2.8309631 , -1.0602154 ]], shape=(2000, 2), dtype=float32)

In [70]:
x = list(model.parameters())

In [71]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [72]:
compiled_model = torch.compile(model)

In [73]:
print(compiled_model)

OptimizedModule(
  (_orig_mod): MyNeuralNetwork()
)


In [76]:
print(model.b)

Parameter containing:
tensor([0.], requires_grad=True)


In [77]:
def mean_squared_error(targets, predictions):
    """ """
    per_sample_loss = torch.square(targets - predictions)
    return torch.mean(per_sample_loss)

In [90]:
def training_step(inputs, targets):
    predictions = model(inputs)
    loss = mean_squared_error(targets, predictions)
    loss.backward()
    optimizer.step()
    model.zero_grad()
    return loss

Similarly, previously we needed to reset parameter gradients by hand by doing
tensor.grad = None on each one. Now we can just do model.zero_grad().
Overall, this may feel a bit confusing—somehow the loss tensor, the optimizer, and the Module instance all seem to be aware of each other through some hidden back-ground mechanism. They’re all interacting with one another via spooky action at a distance. Don’t worry though—you can just treat this sequence of steps **(loss.backward()- optimizer.step() - model.zero_grad()) as a magic incantation to be recited any time you need to write a training step function. Just make sure not to forget model.zero_grad()**. That would be a major bug (and it is unfortunately quite common)!

In [79]:
dir(compiled_model)

['T_destination',
 'W',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_aot_compile',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_call_lazy_check',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_initialize',
 '_is_full_backward_hook',
 '_load_aot_compiled_module',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pre_hooks

In [82]:
import jax

In [ ]:
compiled_model.loss()

In [84]:
dir(compiled_model)

['T_destination',
 'W',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_aot_compile',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_call_lazy_check',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_initialize',
 '_is_full_backward_hook',
 '_load_aot_compiled_module',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pre_hooks

In [85]:
input_dim = 2
output_dim = 1

W = torch.rand(input_dim, output_dim, requires_grad=True)
b = torch.zeros(output_dim, requires_grad=True)

In [87]:
b

tensor([0.], requires_grad=True)

In [88]:
def model1(inputs, W, b):
    return torch.matmul(inputs, W) + b

In [89]:
@torch.compile
def dense(inputs, W, b):
    return torch.nn.relu(torch.matmul(inputs, W) + b)

In [91]:
from torch import nn

model2 = torch.nn.Sequential(
    nn.Linear(input_dim, output_dim), nn.ReLU(), nn.Linear(output_dim, 2)
)

In [2]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x**2

In [3]:
y.backward()

In [4]:
dy_dx = x.grad

In [5]:
print("dy_dx of y=x^2 at x=3.0 is: ", dy_dx)

dy_dx of y=x^2 at x=3.0 is:  tensor(6.)
